### A.5 Implementing multilayer neural networks

In [1]:
import torch
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
 
        self.layers = torch.nn.Sequential(
                
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
 
            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
 
            # output layer
            torch.nn.Linear(20, num_outputs),
        )
 
    def forward(self, x):
        logits = self.layers(x)
        return logits
 

In [2]:
torch.manual_seed(123) # make the random number initialization reproducible for weights and bias
model = NeuralNetwork(50, 3)

In [3]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [11]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [ ]:
print(model.layers[0].weight)

In [ ]:
print(model.layers[0].bias)

In [ ]:
torch.manual_seed(123)
X = torch.rand((1, 50))
with torch.no_grad():
    out = model(X)
print(out)

In [ ]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

In [ ]:
sum = 0
for x in out[0].data:
    sum += x
print(sum)

### A.6 Setting up efficient data loaders

#### **The Dataset class** - The Dataset class is used to instantiate objects that define how each data record is loaded. 

#### **DataLoader** - The DataLoader handles how the data is shuffled and assembled into batches.

In [13]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
 
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [17]:
from torch.utils.data import Dataset
 
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
 
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
 
    def __len__(self):
        return self.labels.shape[0]
 
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
print(train_ds.__getitem__(4))

In [18]:
from torch.utils.data import DataLoader
 
torch.manual_seed(123)
 
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)
 
test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [ ]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

In [20]:
torch.manual_seed(123)
 
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True #having a substantially smaller batch as the last batch in a training epoch can disturb the convergence during training
)

In [ ]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

### Neural network training in PyTorch

#### Model training

In [42]:
import torch.nn.functional as F
 
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
 
num_epochs = 3
 
for epoch in range(num_epochs): 
    
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
 
        logits = model(features)
        
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")
 
    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [ ]:
help(torch.optim)

In [ ]:
print(model.parameters().len())

#### After we trained the model, we can use it to make predictions

In [22]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [23]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [26]:
# returns the index position of the highest value in each row if we set dim=1 
# (setting dim=0 would return the highest value in each column, instead)
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [27]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [28]:
predictions == y_train

tensor([True, True, True, True, True])

In [29]:
torch.sum(predictions == y_train)

tensor(5)

#### generalize the computation of the prediction accuracy

In [31]:
def compute_accuracy(model, dataloader):
 
    model = model.eval()
    correct = 0.0
    total_examples = 0
    
    for idx, (features, labels) in enumerate(dataloader):
        
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)
 
    return (correct / total_examples).item()

In [32]:
compute_accuracy(model, train_loader)

1.0

In [33]:
compute_accuracy(model, test_loader)

1.0

### Saving and loading models

In [35]:
torch.save(model.state_dict(), "my-first-model.pth")

In [ ]:
model = NeuralNetwork(2, 2) 
model.load_state_dict(torch.load("my-first-model.pth"))

### Working with GPUs

In [36]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [ ]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")
print(tensor_1 + tensor_2)

In [40]:
tensor_1 = tensor_1.to("cpu")
tensor_2 = tensor_2.to("cpu")
print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [44]:
tensor_1 = tensor_1.to("mps:0")
tensor_2 = tensor_2.to("mps:1")
print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='mps:0')


#### Comparison of Performance

In [45]:
import torch

a = torch.rand(100, 200)
b = torch.rand(200, 300)

In [46]:
%timeit a @ b

15 μs ± 28.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [47]:
a, b = a.to("mps"), b.to("mps")

In [48]:
%timeit a @ b

22.6 μs ± 736 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
